# IAM Fine-Tuning v5

In [ ]:
!pip install -q transformers peft==0.13.2 accelerate trl datasets
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import files
print("Sube training_data.jsonl:")
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]

In [ ]:
import json
from datasets import Dataset

examples = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        messages = data.get('messages', data.get('conversations', []))
        text = ""
        for msg in messages:
            role = msg['role']
            content = msg['content'][:500]
            if role == 'system':
                text += f"<|system|>\n{content}</s>\n"
            elif role == 'user':
                text += f"<|user|>\n{content}</s>\n"
            elif role == 'assistant':
                text += f"<|assistant|>\n{content}</s>\n"
        examples.append({"text": text})

dataset = Dataset.from_list(examples)
print(f"Ejemplos: {len(dataset)}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Cargando modelo...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Listo!")

In [ ]:
def formatting_func(examples):
    return examples["text"]

training_args = TrainingArguments(
    output_dir="iam_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_steps=100,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=formatting_func,
    args=training_args,
)

print("Entrenando...")
trainer.train()
print("ENTRENAMIENTO COMPLETADO!")

In [ ]:
import shutil
FINAL_PATH = "iam_model/final"
trainer.save_model(FINAL_PATH)
tokenizer.save_pretrained(FINAL_PATH)

shutil.make_archive("iam_model_final", 'zip', FINAL_PATH)

from google.colab import files
files.download("iam_model_final.zip")
print("Descargando...")